In [4]:
import functools

import langchain_mcp_adapters.sessions as lc_sessions
import mcp.client.stdio as mcp_stdio
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage
from langchain_mcp_adapters.client import MultiServerMCPClient
from langgraph.checkpoint.memory import InMemorySaver

# ipykernel 的 sys.stderr 没有真实 fd，不能作为子进程的 stderr
_errlog = open("mcp_stderr.log", "w", encoding="utf-8", buffering=1)
lc_sessions.stdio_client = functools.partial(mcp_stdio.stdio_client, errlog=_errlog)

load_dotenv()
model = init_chat_model("deepseek-flash")
agent = create_agent(model=model,tools=[],system_prompt="你是一个有用的助手",checkpointer=InMemorySaver())
# 定义工具，用MCP获取工具
# 定义mcp client
client = MultiServerMCPClient(
    {
        #本机本地mcp服务,使用uvx下载到本地
        "time": {
            "transport": "stdio",
            "args": [
                "mcp-server-time"
            ],
            "command": "uvx"
        },
        #远程mcp服务,使用http协议走网络调用mcp服务
        "12306-mcp": {
              "transport": "streamable_http",
              "url": "https://mcp.api-inference.modelscope.net/e671f273d07a41/mcp"
        }
    }
)
# 用client拉取tool
tools = await client.get_tools()
agent = create_agent(model = model,tools=tools)
config = {"configurable": {"thread_id":"会话id"}}
# 由于MCP的Tool是异步的，所以必须用ainvoke调用Agent，是异步调用
stream = await agent.astream_events(
    input={ "messages":HumanMessage(content = "现在几点了,我想明天订一张从深圳到海口的车票，"
                                              "帮我看看,你给我所有的建议方案后，我还要一个花销最小的方案和用时最短的方案") },
    version="v3",
    config = {"configurable":{"thread_id":"t1"}}
)
#流式输出
async for mesaages in stream.messages:
    async for text in mesaages.text:
        print(text,end="",flush=True)



我先查一下当前时间、明天日期和深圳、海口的车站信息。直达车次只有一趟，而且全部无票，我来查一下中转方案和其他车站的情况。现在是 **2026-10-29（周四）16:48**（北京时间）。你要买的是**明天 10-30** 深圳→海口的车票。

先说结论：**深圳到海口的直达车只有 Z8006（深圳东 18:02 → 海口 次日 08:48，历时14:46），但明天所有席别都已无票**。所以只能走中转，主要是在广东境内先坐高铁到湛江、茂名，再换乘普速过海。

## 一、明天的全部可行方案

| # | 方案（第一程 + 第二程） | 出发→到达 | 总历时 | 票价/人 | 备注 |
|---|---|---|---|---|---|
| 1 | G5144 深圳07:30→湛江北10:55（二等334）+ K457 湛江西12:17→海口17:45（硬卧133.5） | 07:30 → 当天17:45 | **10:15** | **467.5元** | 全程白天、当天到达；硬卧只剩1张，硬座已无票 |
| 2 | G5268 深圳18:00→茂名南20:36（二等295）+ Z501 茂名22:32→海口06:55（硬座91.5） | 18:00 → 次日06:55 | 12:55 | **386.5元** | 换站换乘，茂名南→茂名；硬座只剩1张 |
| 3 | G5268 深圳18:00→湛江北21:01（二等350）+ K511 湛江西22:09→海口05:35（硬座69.5） | 18:00 → 次日05:35 | 11:35 | 419.5元 | 换站换乘，湛江北→湛江西；硬座有票 |
| 4 | G5164 深圳17:34→茂名南20:46（二等295）+ Z501 茂名22:32→海口06:55（硬座91.5） | 17:34 → 次日06:55 | 13:21 | 386.5元 | 茂名换站 |
| 5 | G5164 深圳17:34→湛江北21:18（二等350）+ Z501 湛江西00:02→海口06:55（硬座69.5，16张） | 17:34 → 次日06:55 | 13:21 | 419.5元 | 湛江换站，第二程票最充足 |
| 6 | G5156 深圳15:18→茂名南18:19（二等295）+ K511 茂名20:53→海口05: